In [1]:
#!pip install statsmodels

In [2]:
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from iso639 import languages
from sklearn.linear_model import LinearRegression
from scipy import stats
from iso639 import languages

In [3]:
base_path = "results"
score_path = "results"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 1
models = [
          #"__scratch__project_462001491__jmnybl__final_embedding_model_checkpoints__v2-20260909-final__final-finetuned-model",
          "Qwen__Qwen3-Embedding-0.6B",
          "Qwen__Qwen3-Embedding-4B",
          "BAAI__bge-m3",
          "microsoft__harrier-oss-v1-0.6b",
          "intfloat__multilingual-e5-large-instruct",
          "google__embeddinggemma-300m",
          "nvidia__llama-embed-nemotron-8b",
          "Octen__Octen-Embedding-8B",
          "codefuse-ai__F2LLM-v2-4B",
          #"ibm-granite__granite-embedding-311m-multilingual-r2",
          ]
filter_prompts=False
dataset = lambda lang: f"mteb__tatoeba-bitext-mining:{lang}" 
#dataset = "mteb__ARCChallenge" 
#dataset = "squad" #"mteb__tatoeba-bitext-mining:fin-eng"#"mteb__ARCChallenge" #"tatoeba:fra-eng" #"mteb__tatoeba-bitext-mining:fin-eng" #"mteb__ARCChallenge"#"mteb__multi-hatecheck:eng" #"mteb__ARCChallenge" #"mteb__tatoeba-bitext-mining:ara-eng" #"mteb__multi-hatecheck:eng" #"mteb__reddit-clustering" #"mteb__stsbenchmark-sts" #"mteb__tatoeba-bitext-mining:fin-eng"
split = "test"
score= f"recall@{k}" # "ndcg@10"#"F1" #"Accuracy" #"V-score" # "average_precision" 
subsplit=""
path = lambda model, lang: f"{base_path}/{model}/{dataset(lang).replace(':','_')}/{split}/{template}_template/"
path_scores = lambda model, lang: f"{score_path}/{model}/{dataset(lang).replace(':','_')}/{split}/{template}_template/"

In [4]:
# Retrieval prompt

def prompts_lang(l):
    l, eng = l.split("-")
    assert eng == "eng"
    try:
        lang = languages.get(part2t=l).name
    except KeyError as err:
        if l == "cmn":
            lang = "Mandarin Chinese"
        else:
            raise(err)
            
    return ["Retrieve parallel sentences.",
            f"Retrieve the corresponding translation in {lang}.",
            f"Given an English sentence, find its translation in {lang}.",
            f"Retrieve parellel sentences in {lang}.",
            f"Translate to {lang}.",
            "Find a sentence that has similar meaning.",
            "Retrieve the corresponding translation.",
            "Given an English sentence, find its translation.",
            "Retrieve parellel sentences.",
            "Translate.",
            "Find a sentence that has similar meaning.",
            ]
prompts_appropriate = prompts_lang

In [24]:
def construct_df(model, lang, show=False, columns_to_select=["recall", "recall_distracted", "prompt_text", "appropriate"]):
    scores_path= path_scores(model, lang)+f"eval@1_2_5_10.json"
    with open(scores_path) as f:
        scores = json.load(f)
    scores_path2= path_scores(model, lang)+f"eval@1_2_5_10_with_distractors.json"
    with open(scores_path2) as f:
        scores2 = json.load(f)
    #with open(path(model,lang)+f"prompt_geometry_10nn_1_distractor_and_1_false_positive.json") as f:
    #    data1 = json.load(f)
    df_scores = pd.DataFrame.from_dict(scores).T
    df_scores["prompt_text"] = [p.rstrip() for p in df_scores["prompt_text"]]
    #print(len(df_scores))
    df_scores2 = pd.DataFrame.from_dict(scores2).T
    df_scores2["prompt_text"] = [p.rstrip() for p in df_scores2["prompt_text"]]
    #print(len(df_scores2))
    assert set(df_scores["prompt_text"].tolist()) == set(df_scores2["prompt_text"].tolist())
    df_all_scores = df_scores.merge(df_scores2, suffixes=("", "_distracted"), on='prompt_text')
    print(len(df_all_scores))
    #df_angle = pd.DataFrame.from_dict(data1).T
    #df = df_all_scores.merge(df_angle, on='prompt_text')
    df = df_all_scores
    df["appropriate"] = [int(p in prompts_appropriate(lang)) for p in df["prompt_text"]]
    df["recall"] = [d["mean"] for d in df["recall@1"]]
    df["recall_distracted"] = [d["mean"] for d in df["recall@1_distracted"]]
    if show: display(df.head())
    if columns_to_select:
        return df[columns_to_select]
    return df

In [26]:
dfs = {}

for m in models:
    dfs_same_model =[]
    for lang in ["cmn-eng", "fin-eng", "vie-eng", "ara-eng", "tur-eng"]:
        try:
            df_lang = construct_df(m, lang)
            df_lang["language"] = lang   # add "source"
            dfs_same_model.append(df_lang)
        except Exception as e:
            print(f"Cannot construct results for {m}")
            raise(e)
    print(len(dfs_same_model))
    dfs[m] = pd.concat(dfs_same_model)

54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5
54
54
56
54
54
56
54
54
56
54
54
56
54
54
56
5


In [27]:
display(dfs["intfloat__multilingual-e5-large-instruct"])

,recall,recall_distracted,prompt_text,appropriate,language
0,0.964,0.028,NO_PROMPT,0,cmn-eng
1,0.963,0.024,EMPTY,0,cmn-eng
2,0.757,0.020,Identify categories in user passages.,0,cmn-eng
3,0.812,0.044,Classify user passages.,0,cmn-eng
4,0.965,0.031,Retrieve text that are semantically similar to...,0,cmn-eng
...,...,...,...,...,...
51,0.977,0.126,"Given an English sentence, find its translation.",1,tur-eng
52,0.984,0.672,Retrieve parellel sentences.,1,tur-eng
53,0.977,0.046,Translate.,1,tur-eng
54,0.979,0.050,Find a sentence that has similar meaning.,1,tur-eng


In [29]:
import statsmodels.formula.api as smf

df = dfs["intfloat__multilingual-e5-large-instruct"]
display(df.head())
model = smf.mixedlm("recall_distracted ~ appropriate", df, groups=df["language"])
result = model.fit()
print(result.summary())

,recall,recall_distracted,prompt_text,appropriate,language
0,0.964,0.028,NO_PROMPT,0,cmn-eng
1,0.963,0.024,EMPTY,0,cmn-eng
2,0.757,0.020,Identify categories in user passages.,0,cmn-eng
3,0.812,0.044,Classify user passages.,0,cmn-eng
4,0.965,0.031,Retrieve text that are semantically similar to...,0,cmn-eng


             Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: recall_distracted
No. Observations: 280     Method:             REML             
No. Groups:       5       Scale:              0.0323           
Min. group size:  56      Log-Likelihood:     77.9231          
Max. group size:  56      Converged:          Yes              
Mean group size:  56.0                                         
-----------------------------------------------------------------
              Coef.   Std.Err.     z      P>|z|   [0.025   0.975]
-----------------------------------------------------------------
Intercept     0.014      0.012    1.131   0.258   -0.010    0.038
appropriate   0.389      0.025   15.298   0.000    0.339    0.439
Group Var     0.000                                              



/users/mynttiam/.local/lib/python3.11/site-packages/statsmodels/base/model.py:589: SingularMatrixWarning: Random effects covariance is singular
  return -self.score(params, *args) / nobs
/users/mynttiam/.local/lib/python3.11/site-packages/statsmodels/base/model.py:589: SingularMatrixWarning: Random effects covariance is singular
  return -self.score(params, *args) / nobs
/users/mynttiam/.local/lib/python3.11/site-packages/statsmodels/base/model.py:589: SingularMatrixWarning: Random effects covariance is singular
  return -self.score(params, *args) / nobs
/users/mynttiam/.local/lib/python3.11/site-packages/statsmodels/base/model.py:589: SingularMatrixWarning: Random effects covariance is singular
  return -self.score(params, *args) / nobs
/tmp/ipykernel_119384/2422836166.py:6: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  result = model.fit()
/tmp/ipykernel_119384/2422836166.py:6: ConvergenceWarning: The Hessian matrix at the estimated parameter values is 